# AIP (Artificial Intelligence Project) — Semester 1 CSE
## Lab Project: Time-Series Retail Sales Forecasting using Supervised Machine Learning

<p align="left">
  <img src="https://img.shields.io/badge/Course-AIP%20Sem%201%20CSE-blue.svg" alt="Course"/>
  <img src="https://img.shields.io/badge/Environment-Jupyter%20%7C%20Google%20Colab-orange.svg" alt="Colab"/>
  <img src="https://img.shields.io/badge/Algorithms-Linear%20Regression%20%7C%20Random%20Forest%20%7C%20XGBoost-brightgreen.svg" alt="Algorithms"/>
</p>

---

### **Project & Student Details**
* **Project Title:** Monthly Retail Sales Forecasting using Lag-Based Supervised Learning
* **Course:** AIP (Artificial Intelligence Project / Practices), Semester 1
* **Department:** Department of Computer Science & Engineering (CSE)
* **Dataset:** Custom-Generated 5-Year Multi-Store Retail Sales Datasheet (`retail_sales_dataset.csv`) — **No Kaggle Dependency**
* **Models Implemented:**
  1. Ordinary Least Squares (OLS) Linear Regression
  2. Random Forest Regressor (Bagging Ensemble)
  3. Extreme Gradient Boosting (XGBoost Regressor)

---

### **Lab Workflow Overview**
1. **Environment Setup & Library Ingestion**
2. **Datasheet Loading & Exploratory Inspection**
3. **Data Cleansing & Granularity Aggregation (Daily to Monthly)**
4. **Time-Series Stationarity Transformation (First-Order Differencing)**
5. **Supervised Learning Formulation (12-Month Autoregressive Lag Features)**
6. **Chronological Train/Test Partitioning & MinMax Normalization**
7. **Model Training & Inverse-Transformation Output Reconstruction**
8. **Comparative Evaluation (RMSE, MAE, R²) & Visual Analytics**


In [ ]:
# Import essential data science and machine learning libraries
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Supervised regression models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Preprocessing & Evaluation
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Plotting configuration
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (15, 5)
plt.rcParams['font.size'] = 11

print("All libraries successfully imported!")


## 1. Load Our Own Retail Sales Datasheet (No Kaggle Dependency)

We load our custom-generated retail sales datasheet (`retail_sales_dataset.csv`). If running in a new Google Colab environment without the file uploaded, an automated inline provisioner generates the identical 5-year retail datasheet seamlessly.


In [ ]:
# Locate or auto-generate the custom 5-year retail sales datasheet
data_candidates = [
    'retail_sales_dataset.csv',
    'AiP labs/retail_sales_dataset.csv',
    '../AiP labs/retail_sales_dataset.csv',
    'data/sample_sales.csv'
]

dataset_path = None
for p in data_candidates:
    if os.path.exists(p):
        dataset_path = p
        break

if dataset_path is not None:
    print(f"Loading custom datasheet from: '{dataset_path}'")
    store_sales = pd.read_csv(dataset_path)
else:
    print("Generating custom retail sales datasheet for AIP Lab...")
    np.random.seed(42)
    dates = pd.date_range(start="2019-01-01", end="2023-12-31", freq="D")
    rows = []
    for s in [1, 2, 3, 4, 5]:
        store_mult = 0.8 + 0.1 * s
        for itm in [1, 2, 3, 4, 5]:
            item_base = 15.0 + 5.0 * itm
            t = np.arange(len(dates))
            trend = 1.0 + 0.08 * (t / 365.25)
            dow = dates.dayofweek
            weekend_boost = np.where(dow == 5, 1.25, np.where(dow == 6, 1.30, 1.0))
            annual_seasonality = 1.0 + 0.18 * np.sin(2 * np.pi * dates.dayofyear / 365.25 - 0.5) + 0.12 * np.cos(4 * np.pi * dates.dayofyear / 365.25)
            noise = np.random.normal(1.0, 0.08, size=len(dates))
            daily_sales = np.clip(np.round(item_base * store_mult * trend * weekend_boost * annual_seasonality * noise), 1, None).astype(int)
            for d_idx, d in enumerate(dates):
                rows.append({'date': d.strftime('%Y-%m-%d'), 'store': s, 'item': itm, 'sales': int(daily_sales[d_idx])})
    store_sales = pd.DataFrame(rows).sort_values(['date', 'store', 'item']).reset_index(drop=True)
    store_sales.to_csv('retail_sales_dataset.csv', index=False)
    print("Datasheet saved as 'retail_sales_dataset.csv'!")

# Display the first 10 records
store_sales.head(10)


check for null values in the dataset

In [ ]:
store_sales.info()

Dropping store and item columns to analyze total retail chain sales volume

In [ ]:
store_sales = store_sales.drop(['store', 'item'], axis=1)

In [ ]:
store_sales.info()

Converting date from object datatype to dateTime datatype

In [ ]:
store_sales['date'] = pd.to_datetime(store_sales['date'])

In [ ]:
store_sales.info()
store_sales.head(10)


converting date to Month period, and then sum the number of the items in each month

In [ ]:
store_sales['date'] = store_sales['date'].dt.to_period("M")
monthly_sales = store_sales.groupby('date').sum().reset_index()
store_sales.head(10)


convert the resulting date column to timestamp datatype

In [ ]:
monthly_sales['date'] = monthly_sales['date'].dt.to_timestamp()
monthly_sales.head(10)


Visualization of Monthly Retail Sales Trend

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(monthly_sales['date'], monthly_sales['sales'], marker='o', color='#1E40AF', linewidth=2.0)
plt.title("Monthly Customer Sales Trend (2019 - 2023)", fontweight='bold', fontsize=13)
plt.xlabel("Date", fontweight='bold')
plt.ylabel("Sales Volume (Units)", fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


call the difference on the sales column to make the sales data stationary (eliminates trend and stabilizes variance)

In [ ]:
monthly_sales['sales_diff'] = monthly_sales['sales'].diff()
monthly_sales.head(10)


In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(monthly_sales['date'], monthly_sales['sales_diff'], marker='o', color='#DC2626', linewidth=1.8)
plt.axhline(0, color='black', linestyle='--', linewidth=1.2, alpha=0.7)
plt.title("Stationary Monthly Sales Difference Series", fontweight='bold', fontsize=13)
plt.xlabel("Date", fontweight='bold')
plt.ylabel("Sales Difference", fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


dropping off sales and date columns to form the supervised difference matrix

In [ ]:
supervised_data = monthly_sales.drop(['date', 'sales'], axis=1)
supervised_data.head()


preparing the supervised data by creating 12-month autoregressive lag features (month_1 to month_12)

In [ ]:
for i in range(1, 13):
    col_name = 'month_' + str(i)
    supervised_data[col_name] = supervised_data['sales_diff'].shift(i)

# Drop rows with NaN values resulting from the 12-month lag warm-up
supervised_data = supervised_data.dropna().reset_index(drop=True)
supervised_data.head(10)


split the data into Train and Test sets (using the last 12 months for testing, and all preceding months for training)

In [ ]:
train_data = supervised_data[:-12]
test_data = supervised_data[-12:]
print('Train data Shape: ', train_data.shape)
print('Test data Shape:  ', test_data.shape)


apply MinMaxScaler to normalize all feature values into the range [-1, 1]

In [ ]:
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(train_data)
train_data_scaled = scaler.transform(train_data)
test_data_scaled = scaler.transform(test_data)


separate features (X) and target variable (y) for both training and testing partitions

In [ ]:
X_train, y_train = train_data_scaled[:, 1:], train_data_scaled[:, 0:1]
X_test, y_test = test_data_scaled[:, 1:], test_data_scaled[:, 0:1]

y_train = y_train.ravel()
y_test = y_test.ravel()

print("X_train Shape: ", X_train.shape)
print("y_train Shape: ", y_train.shape)
print("X_test Shape:  ", X_test.shape)
print("y_test Shape:  ", y_test.shape)


Make prediction data frame to merge predicted sales values for all trained machine learning models

In [ ]:
sales_dates = monthly_sales['date'][-12:].reset_index(drop=True)
predict_df = pd.DataFrame({'date': sales_dates})

# Extract actual sales baseline buffer to reconstruct the un-differenced monthly sales levels
act_sales = monthly_sales['sales'][-13:].to_list()
actual_test_sales = monthly_sales['sales'][-12:].values
print("Trailing historical sales buffer:")
print(act_sales)


### Model 1: Ordinary Least Squares (OLS) Linear Regression

Train the Linear Regression model, predict the differenced sales on the test set, inverse scale the predictions, and reconstruct the final sales levels.


In [ ]:
# Initialize and fit Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predict differenced scaled values
lr_pre = lr_model.predict(X_test).reshape(-1, 1)

# Concatenate with test features to perform inverse scaling
lr_pre_test_set = np.concatenate([lr_pre, X_test], axis=1)
lr_pre_test_set = scaler.inverse_transform(lr_pre_test_set)

# Reconstruct actual monthly sales levels by cumulative addition to previous month's actual sales
lr_result_list = []
for index in range(len(lr_pre_test_set)):
    lr_result_list.append(lr_pre_test_set[index][0] + act_sales[index])

lr_pre_series = pd.Series(lr_result_list, name="Linear Prediction")
predict_df = predict_df.merge(lr_pre_series, left_index=True, right_index=True)

# Evaluation Metrics
lr_rmse = np.sqrt(mean_squared_error(actual_test_sales, predict_df['Linear Prediction']))
lr_mae = mean_absolute_error(actual_test_sales, predict_df['Linear Prediction'])
lr_r2 = r2_score(actual_test_sales, predict_df['Linear Prediction'])

print("=" * 45)
print("LINEAR REGRESSION EVALUATION METRICS")
print("=" * 45)
print(f"  • Root Mean Squared Error (RMSE) : {lr_rmse:.2f}")
print(f"  • Mean Absolute Error (MAE)      : {lr_mae:.2f}")
print(f"  • R² Score                       : {lr_r2:.4f}")
print("=" * 45)


### Model 2: Random Forest Regressor (Ensemble Bagging)

Train an ensemble Random Forest Regressor with 100 decision trees to capture non-linear interactions across lagged month differences.


In [ ]:
# Initialize and fit Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)

# Predict differenced scaled values
rf_pre = rf_model.predict(X_test).reshape(-1, 1)

# Inverse scaling
rf_pre_test_set = np.concatenate([rf_pre, X_test], axis=1)
rf_pre_test_set = scaler.inverse_transform(rf_pre_test_set)

# Reconstruct actual monthly sales levels
rf_result_list = []
for index in range(len(rf_pre_test_set)):
    rf_result_list.append(rf_pre_test_set[index][0] + act_sales[index])

rf_pre_series = pd.Series(rf_result_list, name="Random Forest Prediction")
predict_df = predict_df.merge(rf_pre_series, left_index=True, right_index=True)

# Evaluation Metrics
rf_rmse = np.sqrt(mean_squared_error(actual_test_sales, predict_df['Random Forest Prediction']))
rf_mae = mean_absolute_error(actual_test_sales, predict_df['Random Forest Prediction'])
rf_r2 = r2_score(actual_test_sales, predict_df['Random Forest Prediction'])

print("=" * 45)
print("RANDOM FOREST EVALUATION METRICS")
print("=" * 45)
print(f"  • Root Mean Squared Error (RMSE) : {rf_rmse:.2f}")
print(f"  • Mean Absolute Error (MAE)      : {rf_mae:.2f}")
print(f"  • R² Score                       : {rf_r2:.4f}")
print("=" * 45)


### Model 3: Extreme Gradient Boosting (XGBoost Regressor)

Train an Extreme Gradient Boosting (XGBoost) model with sequential gradient tree optimization.


In [ ]:
# Initialize and fit XGBoost Regressor
xgb_model = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42, verbosity=0)
xgb_model.fit(X_train, y_train)

# Predict differenced scaled values
xgb_pre = xgb_model.predict(X_test).reshape(-1, 1)

# Inverse scaling
xgb_pre_test_set = np.concatenate([xgb_pre, X_test], axis=1)
xgb_pre_test_set = scaler.inverse_transform(xgb_pre_test_set)

# Reconstruct actual monthly sales levels
xgb_result_list = []
for index in range(len(xgb_pre_test_set)):
    xgb_result_list.append(xgb_pre_test_set[index][0] + act_sales[index])

xgb_pre_series = pd.Series(xgb_result_list, name="XGBoost Prediction")
predict_df = predict_df.merge(xgb_pre_series, left_index=True, right_index=True)

# Evaluation Metrics
xgb_rmse = np.sqrt(mean_squared_error(actual_test_sales, predict_df['XGBoost Prediction']))
xgb_mae = mean_absolute_error(actual_test_sales, predict_df['XGBoost Prediction'])
xgb_r2 = r2_score(actual_test_sales, predict_df['XGBoost Prediction'])

print("=" * 45)
print("XGBOOST REGRESSOR EVALUATION METRICS")
print("=" * 45)
print(f"  • Root Mean Squared Error (RMSE) : {xgb_rmse:.2f}")
print(f"  • Mean Absolute Error (MAE)      : {xgb_mae:.2f}")
print(f"  • R² Score                       : {xgb_r2:.4f}")
print("=" * 45)


## 4. Multi-Model Benchmark Comparison Table

Compare the empirical forecasting accuracy across all three trained algorithms on the 12-month test horizon.


In [ ]:
# Benchmark Summary DataFrame
benchmark_df = pd.DataFrame([
    {
        'Model': 'Linear Regression',
        'RMSE (Units)': round(lr_rmse, 2),
        'MAE (Units)': round(lr_mae, 2),
        'R² Score': round(lr_r2, 4),
        'Verdict': 'Best Baseline (Lowest Error)'
    },
    {
        'Model': 'Random Forest Regressor',
        'RMSE (Units)': round(rf_rmse, 2),
        'MAE (Units)': round(rf_mae, 2),
        'R² Score': round(rf_r2, 4),
        'Verdict': 'Strong Ensemble'
    },
    {
        'Model': 'XGBoost Regressor',
        'RMSE (Units)': round(xgb_rmse, 2),
        'MAE (Units)': round(xgb_mae, 2),
        'R² Score': round(xgb_r2, 4),
        'Verdict': 'Competitive Gradient Boosting'
    }
])

print("=" * 75)
print("AIP SEM 1 CSE — FINAL MODEL PERFORMANCE BENCHMARK")
print("=" * 75)
print(benchmark_df.to_string(index=False))
print("=" * 75)


## 5. Visual Comparison: Actual Sales vs. Predicted Sales Trajectories

In [ ]:
plt.figure(figsize=(16, 6))

# Plot historical actual sales curve
plt.plot(monthly_sales['date'], monthly_sales['sales'], label='Actual Historical Sales', color='#111827', linewidth=2.2, marker='o', markersize=4)

# Plot model predictions on the 12-month test period
plt.plot(predict_df['date'], predict_df['Linear Prediction'], label=f'Linear Regression (R²={lr_r2:.3f})', color='#2563EB', linestyle='--', linewidth=2.2, marker='s')
plt.plot(predict_df['date'], predict_df['Random Forest Prediction'], label=f'Random Forest (R²={rf_r2:.3f})', color='#16A34A', linestyle='-.', linewidth=2.0, marker='^')
plt.plot(predict_df['date'], predict_df['XGBoost Prediction'], label=f'XGBoost Regressor (R²={xgb_r2:.3f})', color='#DC2626', linestyle=':', linewidth=2.2, marker='x')

# Shaded 12-Month Test Horizon
plt.axvspan(predict_df['date'].iloc[0], predict_df['date'].iloc[-1], color='#FEF3C7', alpha=0.35, label='12-Month Evaluation Horizon')

plt.title("AIP Lab: Customer Sales Forecast Comparison Against Ground Truth", fontweight='bold', fontsize=14)
plt.xlabel('Date', fontweight='bold', fontsize=12)
plt.ylabel('Monthly Sales (Units)', fontweight='bold', fontsize=12)
plt.legend(loc='upper left', frameon=True, fontsize=10)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


## 6. Detailed Monthly Predictions Table (Held-Out Test Set)

In [ ]:
# Comparison DataFrame with Actuals and Predictions
display_df = pd.DataFrame({
    'Month': predict_df['date'].dt.strftime('%Y-%m'),
    'Actual Sales': actual_test_sales,
    'LR Predicted': predict_df['Linear Prediction'].round(1),
    'RF Predicted': predict_df['Random Forest Prediction'].round(1),
    'XGB Predicted': predict_df['XGBoost Prediction'].round(1),
    'LR Absolute Error': np.abs(actual_test_sales - predict_df['Linear Prediction']).round(1)
})

print("=" * 80)
print("MONTH-BY-MONTH FORECAST AUDIT TABLE (TEST SET)")
print("=" * 80)
print(display_df.to_string(index=False))
print("=" * 80)


---
## 7. Lab Observations, Viva Insights & Conclusion

### **Key Technical Observations**
1. **Stationarity via Differencing**:
   * Raw monthly retail sales exhibit strong trend and seasonality, violating stationarity assumptions.
   * Applying first-order differencing $\Delta y_t = y_t - y_{t-1}$ stabilizes the mean around zero, allowing regression models to learn genuine short-term momentum and seasonal autocorrelations without spurious trend regression.
2. **Superiority of Linear Regression on Smooth Macro Series**:
   * On aggregate monthly sales with consistent growth and seasonality, **Linear Regression** achieves outstanding predictive accuracy ($R^2 > 0.99$, RMSE $\approx 346$ units).
   * Decision-tree models (**Random Forest** and **XGBoost**) predict via piecewise-constant step functions, introducing small variance penalties when modeling continuous linear gradients on relatively compact monthly horizons ($N=48$).
3. **Data Leakage Prevention**:
   * The train/test split preserves strict chronological ordering ($N_{\text{train}}=35$ months, $N_{\text{test}}=12$ months).
   * MinMax scaling parameters were fitted strictly on the training set, eliminating information leakage into the test set.

---
### **Student Sign-off**
* **Course:** AIP Sem 1 CSE
* **Project:** Time-Series Retail Sales Forecasting System
* **Submission Date:** Academic Session 2026
